In [ ]:
import torch
import numpy as np
import librosa
import matplotlib.pyplot as plt
import IPython.display as ipd
from IPython.display import display, HTML

from sincnet.model import SincNet
from sincnet.mulaw import MuLawQuant, mu_law_compand

from datasets.utils.waveform import WaveformLoader
from training.utils.stft import TorchSTFT

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
SAMPLE_RATE = 16_000
wav_np, _ = librosa.load("audio/invertibility/p232_001.wav", sr=SAMPLE_RATE, mono=True)
wav_np = wav_np / (np.abs(wav_np).max() + 1e-8)  # peak normalize
wav = torch.from_numpy(wav_np).unsqueeze(0).to(device)  # (1, T)
print(f"waveform: {wav.shape},  duration: {wav.shape[-1] / SAMPLE_RATE:.2f}s")
display(ipd.Audio(wav_np, rate=SAMPLE_RATE))

In [ ]:
params = {
    "fs": SAMPLE_RATE,
    "fps": 128,
    "n_bins": 128 * 4,
    "scale": "mel",
    "component": "complex",
    "causal": False,
    "decoder_type": "learnt",
    #"apply_sinc_envelope": True
}

sinc : SincNet = (
    SincNet(**params)
    .eval()
    .to(device)
)


weight_dir = "pretrained/"
#weight_dir = "training/trainings/gtzan/ckpt"
sinc.load_pretrained_weights(weight_dir, verbose=True)

In [ ]:
n_fft = 1024
win_length = 1024
hop_length = win_length // 4
stft = TorchSTFT(sample_rate=params["fs"], n_fft=n_fft, hop_length=hop_length, win_length=win_length).to(device)

## 1 - Invertibility

In [ ]:
with torch.no_grad():
    spec  = sinc.encode(wav)                         # (1, 2, F, T)
    recon = sinc.decode(spec, length=wav.shape[-1])  # (1, T)

snr = 10 * torch.log10(
    wav.pow(2).sum() / (wav - recon).pow(2).sum()
).item()
print(f"SNR (exact decoder): {snr:.1f} dB")

waveform = audio_loader.load_segment(audio_path, offset=15, duration=5, nchannels=1)
loudness = audio_loader.measure_loudness(waveform)
waveform = audio_loader.normalise_loudness(waveform, loudness, target_lufs=-23)

plt.plot(waveform[0])
ipd.Audio(waveform, rate=SAMPLE_RATE, autoplay=False)

In [ ]:
eps=1e-8
q_bits = 8
quantizer = MuLawQuant(q_bits=q_bits)


with torch.no_grad():    
    # trnsform the waveform into tensor
    wav = torch.from_numpy(waveform).to(device).float()
    print("wav_in", wav.min(), wav.max(), wav.shape)

    #encode and decode waveform
    spec = s = sinc.encode(wav)
    #s[:, :, 96:] = 0
    reconstructed_wav = sinc.decode(s)

    # predictions = sinc(wav)
    # reconstructed_wav = predictions["waveform"]

    print("wav_ou", reconstructed_wav.min(), reconstructed_wav.max(), reconstructed_wav.shape)
    print("pr-spec:", s.min(), s.max())

    #apply quantization
    sq, scale = quantizer.quantize(s)
    print("cq-spec:", sq.min(), sq.max())
    sr = quantizer.dequantize(sq, scale)
    print("dq-spec:", sr.min(), sr.max())
    detokenized_wav = sinc.decode(sr)

    #apply mu-law-companding for visualization
    s = mu_law_compand(s, q_bits=q_bits)
    sr = mu_law_compand(sr, q_bits=q_bits)

    m = torch.sqrt(s[:,0]**2 + s[:,1]**2) if s.shape[1] > 1 else s[:,0].abs()
    mr = torch.sqrt(sr[:,0]**2 + sr[:,1]**2) if sr.shape[1] > 1 else sr[:,0].abs()

    # reduce to 2D for visualization (average over channels mono/stereo and frequencies)
    s = s.flatten(1,2).mean(dim=0)
    sr = sr.flatten(1,2).mean(dim=0)


f, axes = plt.subplots(2, 2, figsize=(10, 6), sharex="all")

y = s.detach().numpy()
yr = sr.detach().numpy()

axes[0,0].imshow(y)
axes[1,0].imshow(np.abs(y))
#axes[2,0].imshow(m[0].detach().numpy())

axes[0,1].imshow(yr)
axes[1,1].imshow(np.abs(yr))
#axes[2,1].imshow(mr[0].detach().numpy())

# Create your Audio objects
audio1 = ipd.Audio(reconstructed_wav, rate=SAMPLE_RATE, autoplay=False)
audio2 = ipd.Audio(detokenized_wav, rate=SAMPLE_RATE, autoplay=False)
# Embed them in HTML with flexbox for horizontal layout
html_content = f"""
<div style="display:flex; justify-content:space-around;">
    <div>
        <h3>Reconstructed waveform</h3>
        {audio1._repr_html_()}
    </div>
    <div>
        <h3>Dequantized waveform</h3>
        {audio2._repr_html_()}
    </div>
</div>
"""
# Display the HTML
display(HTML(html_content))

In [ ]:
def compute_log1p_stft(waveform:torch.Tensor, n_bits:int=8) -> torch.Tensor:
    """compute log1p stft and return spectrogram"""
    spectrograms = stft.compute_log1p_magnitude(waveform, n_bits=n_bits).unsqueeze(0)
    return spectrograms

stft_wav = compute_log1p_stft(wav)
stft_rec = compute_log1p_stft(reconstructed_wav)
diff = stft_rec - stft_wav

f, axes = plt.subplots(1, 3, figsize=(15, 20))
axes[0].imshow(stft_wav[0,0,0])
axes[1].imshow(stft_rec[0,0,0])
axes[2].imshow(diff[0,0,0])

axes[0].set_title("STFT(x)")
axes[1].set_title("STFT(decode(encode(x)))")
axes[2].set_title("difference")

In [ ]:
mag = sinc.magnitude(spec)[0].cpu().numpy()  # (F, T)

fig, ax = plt.subplots(figsize=(12, 4))
ax.imshow(np.log1p(mag), origin="lower", aspect="auto", cmap="inferno")
ax.set_title("iSincNet magnitude spectrogram (mel-256, exact decoder)")
ax.set_xlabel("Frame")
ax.set_ylabel("Bin")
plt.tight_layout()
plt.show()

In [ ]:
N_FFT, HOP = 1024, 256

def stft_mag(x: torch.Tensor) -> np.ndarray:
    S = torch.stft(
        x.squeeze(0), n_fft=N_FFT, hop_length=HOP, win_length=N_FFT,
        window=torch.hann_window(N_FFT, device=x.device),
        return_complex=True,
    )
    return S.abs().cpu().numpy()

S_orig  = stft_mag(wav)
S_recon = stft_mag(recon)
vmax    = float(np.log1p(S_orig).max())

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
kw = dict(origin="lower", aspect="auto", cmap="magma", vmin=0, vmax=vmax)
axes[0].imshow(np.log1p(S_orig),  **kw)
axes[0].set_title("Original")
axes[1].imshow(np.log1p(S_recon), **kw)
axes[1].set_title(f"Reconstructed  ({snr:.1f} dB)")
for ax in axes:
    ax.set_xlabel("Frame")
    ax.set_ylabel("Freq bin")
plt.suptitle("STFT comparison: original vs iSincNet encode -> exact decode")
plt.tight_layout()
plt.show()

## 2 - Mu-law quantization

In [ ]:
audio_pathes = {
    0:"audio/stems/stem_pad.wav",
    1:"audio/stems/stem_drum1.wav",
    2:"audio/stems/stem_drum2.wav",

}

selection = [0,2]

wavs = []
for choice in selection:
    audio_path = audio_pathes[choice]
    waveform = audio_loader.load_segment(audio_path, offset=0, duration=5, nchannels=1)
    wavs.append(waveform)

mix = sum(wavs)
wavs.insert(0, mix)

plt.plot(mix[0])
ipd.Audio(mix, rate=SAMPLE_RATE, autoplay=False)

In [ ]:
spectrograms = []
reconstructed = []

with torch.no_grad():
    q, scale  = quantizer.quantize(spec)
    spec_dq   = quantizer.dequantize(q, scale)
    recon_q   = sinc.decode(spec_dq, length=wav.shape[-1])

snr_q = 10 * torch.log10(
    wav.pow(2).sum() / (wav - recon_q).pow(2).sum()
).item()
print(f"SNR after {Q_BITS}-bit mu-law quantization: {snr_q:.1f} dB")

recon_q_np = recon_q.squeeze(0).cpu().numpy()
display(HTML(f"""
<div style="display:flex; gap:40px; align-items:center;">
  <div><b>Original</b><br>{ipd.Audio(wav_np, rate=SAMPLE_RATE)._repr_html_()}</div>
  <div><b>After {Q_BITS}-bit mu-law ({snr_q:.1f} dB)</b><br>{ipd.Audio(recon_q_np, rate=SAMPLE_RATE)._repr_html_()}</div>
</div>
"""))

In [ ]:
## 3 - Griffin-Lim (magnitude-only reconstruction)

In [ ]:
# Discard phase: keep only the magnitude, then iteratively estimate it with Griffin-Lim.
# This is the reconstruction you get when you have no phase information (e.g. from a mel filterbank).
N_ITERS = 50
with torch.no_grad():
    mag_only = sinc.magnitude(spec)              # (1, F, T) - phase discarded
    spec_gl  = sinc.griffin_lim(mag_only, n_iters=N_ITERS)
    recon_gl = sinc.decode(spec_gl, length=wav.shape[-1])

snr_gl = 10 * torch.log10(
    wav.pow(2).sum() / (wav - recon_gl).pow(2).sum()
).item()
print(f"SNR Griffin-Lim ({N_ITERS} iters): {snr_gl:.1f} dB")

recon_gl_np = recon_gl.squeeze(0).cpu().numpy()
display(HTML(f"""
<div style="display:flex; gap:40px; align-items:center;">
  <div><b>Original</b><br>{ipd.Audio(wav_np, rate=SAMPLE_RATE)._repr_html_()}</div>
  <div><b>Griffin-Lim {N_ITERS} iters ({snr_gl:.1f} dB)</b><br>{ipd.Audio(recon_gl_np, rate=SAMPLE_RATE)._repr_html_()}</div>
  <div><b>Exact ({snr:.1f} dB)</b><br>{ipd.Audio(recon_np, rate=SAMPLE_RATE)._repr_html_()}</div>
</div>
"""))

## 4 - Linearity

In [ ]:
def load_wav(path: str) -> torch.Tensor:
    y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    y = y / (np.abs(y).max() + 1e-8)
    return torch.from_numpy(y).unsqueeze(0).to(device)

stem_a = load_wav("audio/stems/stem_pad.wav")
stem_b = load_wav("audio/stems/stem_drum1.wav")

n = min(stem_a.shape[-1], stem_b.shape[-1])
stem_a, stem_b = stem_a[..., :n], stem_b[..., :n]

with torch.no_grad():
    mix_direct = sinc.encode(stem_a + stem_b)
    mix_linear = sinc.encode(stem_a) + sinc.encode(stem_b)

err = (mix_direct - mix_linear).abs().max().item()
print(f"max |encode(a+b) - (encode(a)+encode(b))| = {err:.2e}  (machine precision -> linear)")